# 03. Pruebas con Qwen

Se comparan **seis modelos Qwen** con los casos `C001` a `C008` de `casos2.json`, en español e inglés (**96 pruebas**).

Las respuestas se agregan a `resultados.csv`, sin borrar las pruebas de las otras familias ni las pruebas piloto.

## 1. Cargar los casos, los modelos y resultados previos

In [1]:
import os
import csv
import json
import time
from copy import deepcopy

import requests

archivo_resultados = "resultados.csv"
url_ollama = "http://localhost:11434/api"
num_ctx = 8192
num_predict = 4096

columnas = [
    "id_caso", "id_modelo", "idioma", "intento", "solicitud_json", "estado",
    "tiempo_s", "respuesta", "error", "evaluacion", "motivo_evaluacion"
]

# Cargar los ocho casos
with open("casos2.json", "r", encoding="utf-8-sig") as f:
    datos = json.load(f)

casos = {c["id_caso"]: c for c in datos["casos"]}
ids_casos = [f"C{i:03d}" for i in range(1, 9)]
idiomas = ["es", "en"]

# Probar los seis Qwen de menor a mayor tamaño
ids_modelos = ["M08", "M01", "M04", "M13", "M15", "M19"]
with open("inventario_22_llm.csv", "r", encoding="utf-8-sig", newline="") as f:
    modelos_por_id = {m["id_modelo"]: m for m in csv.DictReader(f)}

modelos_a_probar = [modelos_por_id[i] for i in ids_modelos]

# Leer resultados previos o crear el CSV si no existe
if os.path.exists(archivo_resultados):
    with open(archivo_resultados, "r", encoding="utf-8-sig", newline="") as f:
        registros = list(csv.DictReader(f))
else:
    with open(archivo_resultados, "w", encoding="utf-8-sig", newline="") as f:
        csv.DictWriter(f, fieldnames=columnas).writeheader()
    registros = []

for m in modelos_a_probar:
    print(m["id_modelo"], "|", m["nombre_modelo"])
print("Casos:", len(ids_casos), "| Idiomas:", len(idiomas))
print("Pruebas previstas:", len(modelos_a_probar) * len(ids_casos) * len(idiomas))
print("Registros anteriores:", len(registros))

M08 | Qwen3.5 4B
M01 | Qwen2.5 7B
M04 | Qwen3.5 9B
M13 | Qwen2.5 14B Instruct
M15 | Qwen3.8 27B
M19 | Qwen3 30B A3B Instruct 2507
Casos: 8 | Idiomas: 2
Pruebas previstas: 96
Registros anteriores: 263


## 2. Preparar las solicitudes

Todos los Qwen reciben el mismo prompt, la predicción de coautoría y las publicaciones en español e inglés.

In [2]:
instrucciones = """Redacta una justificación semántica breve de la predicción de coautoría recibida, con un máximo de 180 palabras.

Responde únicamente en tres apartados:

1. Predicción: indica la clasificación recibida. Si hay probabilidad y umbral, menciona sus valores.
2. Coincidencias y diferencias: explica los temas o métodos de las publicaciones de ambos autores y cita sus artículos.
3. Limitaciones: indica qué información falta o qué afirmaciones no pueden respaldarse.

Reglas:
- La predicción ya fue calculada. No intentes explicar cómo la obtuvo el predictor.
- Utiliza la clasificación conjunta cuando exista; en caso contrario, utiliza la clasificación por umbral indicada para el método.
- Si faltan probabilidades, indícalo sin dejar de comparar las publicaciones.
- La similitud entre artículos no es una probabilidad de coautoría.
- Una similitud indefinida no significa que no existan coincidencias.
- No inventes información ni afirmes que las coincidencias causaron la predicción.
- No repitas las instrucciones ni copies los resúmenes completos."""

def preparar_solicitud(caso, idioma, etiqueta):
    entrada = deepcopy(caso["entrada_llm"])
    umbral_general = datos["regla_coautoria"].get("umbral_definitivo")

    for metodo in entrada["prediccion_coautoria"]["resultados_por_metodo"]:
        p = metodo.get("probabilidad_coautoria")
        umbral = metodo.get("umbral_aplicado")
        if umbral is None:
            umbral = umbral_general
        metodo["umbral_aplicado"] = umbral

        if p is not None and umbral is not None:
            metodo["clasificacion_por_umbral"] = "favorable" if p >= umbral else "desfavorable"
        else:
            metodo["clasificacion_por_umbral"] = None

    instruccion_idioma = datos["idiomas"]["instruccion_por_idioma"][idioma]
    return {
        "model": etiqueta,
        "messages": [
            {"role": "system", "content": instrucciones + "\n" + instruccion_idioma},
            {"role": "user", "content": json.dumps(entrada, ensure_ascii=False, indent=2)}
        ],
        "stream": False,
        "think": False,
        "options": {"temperature": 0, "num_ctx": num_ctx, "num_predict": num_predict}
    }


def ya_completa(id_modelo, id_caso, idioma, solicitud):
    # Solo omitir si ya se completó con este mismo prompt y parámetros
    # (así se repiten las pruebas piloto de Qwen2.5 7B que tenían otro prompt)
    for fila in registros:
        if (fila["id_modelo"] == id_modelo
                and fila["id_caso"] == id_caso
                and fila["idioma"] == idioma
                and fila["estado"] == "completa"):
            try:
                anterior = json.loads(fila["solicitud_json"])
                if (anterior.get("messages") == solicitud["messages"]
                        and anterior.get("options") == solicitud["options"]):
                    return True
            except Exception:
                pass
    return False

## 3. Consultar a Ollama

Se guarda la respuesta, el tiempo y el estado de cada intento. Si el modelo no acepta `think=False`, se reintenta sin ese campo.

In [3]:
def pedir_justificacion(solicitud):
    inicio = time.time()
    enviada = solicitud
    resultado = {"estado": "error", "respuesta": "", "error": ""}

    try:
        r = requests.post(url_ollama + "/chat", json=enviada, timeout=1800)

        # Si el modelo no acepta 'think', reintentar sin ese campo
        if r.status_code == 400 and "think" in r.text.lower():
            enviada = solicitud.copy()
            enviada.pop("think")
            r = requests.post(url_ollama + "/chat", json=enviada, timeout=1800)

        r.raise_for_status()
        salida = r.json()
        texto = salida.get("message", {}).get("content", "")
        resultado["respuesta"] = texto

        if salida.get("done_reason") == "length":
            resultado["estado"] = "truncada"
        elif salida.get("done") and texto.strip() != "":
            resultado["estado"] = "completa"
        else:
            resultado["estado"] = "incompleta"

    except Exception as e:
        resultado["error"] = str(e)

    resultado["tiempo_s"] = round(time.time() - inicio, 2)
    return resultado, enviada

## 4. Ejecutar las 96 pruebas

Se ejecuta un modelo a la vez, empezando por el más pequeño.

In [4]:
# Revisar modelos Qwen instalados en Ollama
r = requests.get(url_ollama + "/tags", timeout=15)
r.raise_for_status()
instalados = [m["name"].lower() for m in r.json()["models"]]

faltantes = []
nuevas = 0

for modelo in modelos_a_probar:
    id_modelo = modelo["id_modelo"]
    etiqueta = modelo["etiqueta_ollama"]
    nombre = modelo["nombre_modelo"]

    if etiqueta.lower() not in instalados:
        faltantes.append(etiqueta)
        print("NO INSTALADO:", id_modelo, "|", etiqueta)
        continue

    print(f"\nMODELO: {id_modelo} | {nombre}")

    for id_caso in ids_casos:
        for idioma in idiomas:
            solicitud = preparar_solicitud(casos[id_caso], idioma, etiqueta)
            if ya_completa(id_modelo, id_caso, idioma, solicitud):
                print("  ", id_caso, idioma, "| ya completada")
                continue

            # Contar intentos previos para este modelo, caso e idioma
            intento = 1
            for f in registros:
                if f["id_modelo"] == id_modelo and f["id_caso"] == id_caso and f["idioma"] == idioma:
                    intento += 1

            resultado, enviada = pedir_justificacion(solicitud)

            fila = {
                "id_caso": id_caso,
                "id_modelo": id_modelo,
                "idioma": idioma,
                "intento": intento,
                "solicitud_json": json.dumps(enviada, ensure_ascii=False),
                "estado": resultado["estado"],
                "tiempo_s": resultado["tiempo_s"],
                "respuesta": resultado["respuesta"],
                "error": resultado["error"],
                "evaluacion": "",
                "motivo_evaluacion": ""
            }

            with open(archivo_resultados, "a", encoding="utf-8", newline="") as f:
                csv.DictWriter(f, fieldnames=columnas).writerow(fila)

            registros.append(fila)
            nuevas += 1

            print("  ", id_caso, idioma, "|", fila["estado"], "|", fila["tiempo_s"], "s")
            if fila["error"]:
                print("     Error:", fila["error"][:180])

    # Liberar el modelo de memoria antes de cargar el siguiente
    try:
        requests.post(url_ollama + "/generate", json={"model": etiqueta, "keep_alive": 0}, timeout=60)
    except Exception:
        pass

print("\nIntentos nuevos guardados:", nuevas)
print("Total de filas en resultados.csv:", len(registros))
if faltantes:
    print("\nModelos pendientes de instalar:")
    for etiqueta in faltantes:
        print("  ollama pull", etiqueta)


MODELO: M08 | Qwen3.5 4B
   C001 es | completa | 30.64 s
   C001 en | completa | 4.72 s
   C002 es | completa | 5.08 s
   C002 en | completa | 5.06 s
   C003 es | completa | 4.37 s
   C003 en | completa | 4.43 s
   C004 es | completa | 4.65 s
   C004 en | completa | 5.3 s
   C005 es | completa | 4.55 s
   C005 en | completa | 4.46 s
   C006 es | completa | 4.37 s
   C006 en | completa | 5.15 s
   C007 es | completa | 4.98 s
   C007 en | completa | 4.84 s
   C008 es | completa | 4.81 s
   C008 en | completa | 4.42 s

MODELO: M01 | Qwen2.5 7B
   C001 es | completa | 12.35 s
   C001 en | completa | 4.56 s
   C002 es | completa | 5.77 s
   C002 en | completa | 4.61 s
   C003 es | completa | 5.45 s
   C003 en | completa | 4.81 s
   C004 es | completa | 5.37 s
   C004 en | completa | 4.4 s
   C005 es | completa | 5.53 s
   C005 en | completa | 4.78 s
   C006 es | completa | 5.21 s
   C006 en | completa | 5.1 s
   C007 es | completa | 6.16 s
   C007 en | completa | 5.14 s
   C008 es | comple

## 5. Revisar el avance

Se cuentan únicamente las respuestas completas con el prompt y parámetros de esta etapa.

In [5]:
completas = 0
for m in modelos_a_probar:
    for id_caso in ids_casos:
        for idioma in idiomas:
            solicitud = preparar_solicitud(casos[id_caso], idioma, m["etiqueta_ollama"])
            if ya_completa(m["id_modelo"], id_caso, idioma, solicitud):
                completas += 1

total = len(modelos_a_probar) * len(ids_casos) * len(idiomas)
print(f"Avance: {completas} de {total} respuestas completas")
print("Pendientes:", total - completas)
print("Archivo:", archivo_resultados)

Avance: 96 de 96 respuestas completas
Pendientes: 0
Archivo: resultados.csv
